autotunenet pilot 1 of 1

Generated by `autotunenet prepare --runtime kaggle`. Runs part of the
AutoTuneNet benchmark experiment `pilot` on Kaggle's GPU T4 x2:
one group of cells per GPU, in parallel.

- installs: `autotunenet[bench] @ git+https://github.com/mitishraina/AutoTuneNet.git@5e30bd96296e8e23c8507426abcd571bbe9fdadd`
- cells: 4 (2 + 2 per GPU)

Steps: install, write the experiment (and any plugins), link the attached
datasets, run, and pack the results into `autotunenet_results.tar.gz`.

In [ ]:
import glob
import json
import os
import subprocess
import tarfile
import time

JOB = json.loads(r'''{"install": "autotunenet[bench] @ git+https://github.com/mitishraina/AutoTuneNet.git@5e30bd96296e8e23c8507426abcd571bbe9fdadd", "experiment": "pilot", "experiment_yaml": "# Short timing pilot on hosted GPUs: per-epoch cost of a single model and of PBT\n# on both tasks, used to size the shards for the full comparison run.\nname: pilot\nepochs: 10\nseeds: [42]\ntasks: [mnist, cifar10]\n\nscenarios:\n  stationary: {}\n\nmethods:\n  fixed_lr: {}\n  pbt: {population: 4, perturbation_interval: 2}\n", "plugins": {}, "gpu_cells": [[{"task": "cifar10", "scenario": "stationary", "method": "fixed_lr", "seed": 42}, {"task": "mnist", "scenario": "stationary", "method": "fixed_lr", "seed": 42}], [{"task": "cifar10", "scenario": "stationary", "method": "pbt", "seed": 42}, {"task": "mnist", "scenario": "stationary", "method": "pbt", "seed": 42}]]}''')
WORK = "/kaggle/temp/autotunenet"          # outside /kaggle/working, so not part of the output
DATA = os.path.join(WORK, "data")
RESULTS = "/kaggle/working/results"
LOGS = "/kaggle/working/logs"
for folder in (WORK, DATA, LOGS):
    os.makedirs(folder, exist_ok=True)
START = time.time()


def sh(cmd, **kwargs):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, **kwargs)

## 1. Install AutoTuneNet and write the experiment

In [ ]:
sh(f"pip install -q '{JOB['install']}'")
experiment = os.path.join(WORK, "experiment.yaml")
open(experiment, "w").write(JOB["experiment_yaml"])
plugins = []
for name, source in JOB["plugins"].items():
    path = os.path.join(WORK, name)
    open(path, "w").write(source)
    plugins += ["--plugins", path]
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")

## 2. Data
Each attached `autotunenet-<task>` dataset holds the files its task reads, relative to the data
directory. Kaggle may keep the uploaded archive or unpack it; both are handled.

In [ ]:
for archive in glob.glob("/kaggle/input/**/*.tar", recursive=True):
    with tarfile.open(archive) as tar:
        tar.extractall(DATA, filter="data")
for marker in glob.glob("/kaggle/input/**/autotunenet_data.json", recursive=True):
    root = os.path.dirname(marker)
    for entry in os.listdir(root):
        target = os.path.join(DATA, entry)
        if entry != "autotunenet_data.json" and not os.path.exists(target):
            os.symlink(os.path.join(root, entry), target)
print("data:", sorted(os.listdir(DATA)))
ENV = {**os.environ, "AUTOTUNENET_DATA_DIR": DATA}

## 3. Run one group of cells per GPU

In [ ]:
procs = []
for gpu, cells in enumerate(JOB["gpu_cells"]):
    if not cells:
        continue
    cells_file = os.path.join(WORK, f"cells_gpu{gpu}.json")
    json.dump(cells, open(cells_file, "w"))
    cmd = ["autotunenet", "run", "--experiment", experiment, "--cells", cells_file,
           "--results-dir", RESULTS, "--device", "cuda", "--no-report"] + plugins
    log_path = os.path.join(LOGS, f"gpu{gpu}.log")
    print(f"GPU {gpu}: {len(cells)} cells", flush=True)
    procs.append((gpu, log_path, subprocess.Popen(
        cmd, cwd=WORK, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)},
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT)))
failed = [gpu for gpu, _, p in procs if p.wait() != 0]
print(f"finished in {(time.time() - START) / 60:.1f} min; GPUs with failed cells: {failed or 'none'}")

## 4. Package the results

In [ ]:
with tarfile.open("/kaggle/working/autotunenet_results.tar.gz", "w:gz") as tar:
    if os.path.isdir(RESULTS):
        tar.add(RESULTS, arcname="results")
    tar.add(LOGS, arcname="logs")
for gpu, log_path, _ in procs:
    print(f"--- last lines of GPU {gpu} ---")
    print("".join(open(log_path).readlines()[-15:]))